<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 9 — Statistics and Plotting Reference

A lookup guide. Notebooks 01 and 02 are meant to be read in order; this one is
meant to be **searched**. Keep it open while you work on Homework 8.

Every section runs on the small table `cart` defined in the first cell (the
coffee cart of notebook 01), so you can change any example and see what
happens. Run the setup cell, then jump to the section you need.

| Section | When you need it |
|---|---|
| [A — Summary numbers](#A) | mean, median, variance, standard deviation |
| [B — Quantiles, fences, outliers](#B) | quartiles, the 1.5 × iqr rule, flagging |
| [C — One summary per group](#C) | the same numbers for every weekday |
| [D — Two columns: correlation](#D) | r, and the index trap |
| [E — Making a figure](#E) | Figure and Axes, size, several Axes |
| [F — One chart per kind of question](#F) | histogram, bars, scatter, line, box plot |
| [G — Labels, titles, legends, text](#G) | everything a reader needs |
| [H — Scales, limits, ticks](#H) | logarithmic axes, where an axis starts, dates |
| [I — Colours, markers, line styles](#I) | and when to leave them alone |
| [J — Saving and closing](#J) | files, memory, and why there is no `show()` |
| [K — Asking an Axes what is on it](#K) | reading a figure back in code |
| [L — The pandas shortcut](#L) | `series.plot(ax=ax)` |
| [M — Checklist and common mistakes](#M) | before anyone else sees it |

This guide uses matplotlib's **object interface** throughout: make an Axes,
then call its methods. Session 8's reference
(`../08-Python-Data-Analysis/03_Data_Analysis_Reference`) covers the pandas
underneath: missing values, `groupby`, dates.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

rng = np.random.default_rng(84)
days = pd.date_range("2026-06-01", periods=84, freq="D")
temperature = np.round(18 + 8 * np.sin(np.arange(84) / 13) + rng.normal(0, 3, 84), 1)
weekend = days.dayofweek >= 5
cups = np.round(260 - 5.5 * temperature + 60 * weekend + np.arange(84) * 0.9
                + rng.normal(0, 18, 84)).astype(float)

cart = pd.DataFrame({"date": days, "weekday": days.day_name(), "weekend": weekend,
                     "temperature": temperature, "cups": cups})
cart.loc[[5, 40], "cups"] = np.nan          # two days the count was not written down

ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cart.head(8)

---

<a id="A"></a>
## A — Summary numbers

All of these skip missing values in pandas.

| You want | Write this | Notes |
|---|---|---|
| how many values are not missing | `s.count()` | `len(s)` counts missing ones too |
| the mean | `s.mean()` | sum ÷ count |
| the median | `s.median()` | middle value; mean of the middle two for an even count |
| the sample variance | `s.var()` | divides by count − 1 |
| the sample standard deviation | `s.std()` | square root of the above |
| smallest, largest | `s.min()`, `s.max()` | |
| the sum | `s.sum()` | |
| most of the above at once | `s.describe()` | count, mean, std, min, quartiles, max |
| several of your choice | `s.agg(["mean", "median", "std"])` | |
| the most frequent value | `s.mode()` | a Series: there can be ties |
| how often each value occurs | `s.value_counts()` | add `dropna=False` to count the blanks |

In [ ]:
cups = cart["cups"]
print("len", len(cups), " count", cups.count())
print(cups.agg(["mean", "median", "var", "std", "min", "max"]))

### Variance: pandas and numpy disagree

| Call | Divides by | Name |
|---|---|---|
| `s.var()`, `s.std()` | count − 1 | sample |
| `s.var(ddof=0)`, `s.std(ddof=0)` | count | population |
| `np.var(a)`, `np.std(a)` | count | population |
| `np.var(a, ddof=1)`, `np.std(a, ddof=1)` | count − 1 | sample |

`ddof` is what is subtracted from the count. Say which one you report.

In [ ]:
known = cups.dropna()
print("pandas  s.var()          ", known.var())
print("numpy   np.var(a)        ", np.var(known.to_numpy()))
print("numpy   np.var(a, ddof=1)", np.var(known.to_numpy(), ddof=1))

### Missing values in numpy

A numpy function on an array with a missing value returns `nan`. Use the
pandas method, or drop the missing values first, or use the `nan...` variant.

In [ ]:
array = cups.to_numpy()
print("np.mean(array)     ", np.mean(array))
print("np.nanmean(array)  ", np.nanmean(array))
print("cups.mean()        ", cups.mean())

### From the definitions

```python
known = s.dropna()
count = len(known)
mean = known.sum() / count
variance = ((known - mean) ** 2).sum() / (count - 1)
std = variance ** 0.5
```

### When a number does not exist

| Case | pandas returns |
|---|---|
| mean, median, min, max of no values | `nan` |
| variance or standard deviation of one value | `nan` |
| sum of no values | `0` |

Test for it with `np.isnan(x)` or `pd.isna(x)`. `x == np.nan` is always False.

---

<a id="B"></a>
## B — Quantiles, fences, outliers

| You want | Write this |
|---|---|
| the first quartile | `s.quantile(0.25)` |
| the three quartiles | `s.quantile([0.25, 0.5, 0.75])` |
| the 90th percentile | `s.quantile(0.9)` |
| the same in numpy | `np.quantile(a, 0.25)`; `np.nanquantile` when values are missing |
| the interquartile range | `s.quantile(0.75) - s.quantile(0.25)` |
| the rank of each value | `s.rank()` |

In [ ]:
q1, q2, q3 = cups.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
print("q1", q1, " median", q2, " q3", q3, " iqr", iqr)

A quantile that falls between two values is **interpolated**. The default is
linear: the 0.25 quantile of `[10, 20, 30, 40]` is at position 0.25 × (4 − 1) =
0.75, three quarters of the way from 10 to 20, which is 17.5. Other rules
exist (`interpolation="lower"`, `"nearest"`, ...) and give other answers on
small data. Use the default unless told otherwise, in both libraries.

### Fences and flags

```python
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outside = (s < lower_fence) | (s > upper_fence)      # True/False per row
```

In [ ]:
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outside = (cups < lower_fence) | (cups > upper_fence)

print("fences", lower_fence, upper_fence)
print("flagged:", int(outside.sum()), " rows in the table:", len(cart))

| You want | Write this |
|---|---|
| how many are flagged | `outside.sum()` |
| what share | `outside.mean()` |
| the flagged rows, to look at | `table[outside]` |
| the rest, **as a new table** | `table[~outside]` |
| a flag column in the table | `table.assign(outlier=outside)` |

- Parentheses around each comparison are required: `|` binds tighter than `<`.
- A comparison with a missing value is `False`, so a missing value is never
  flagged.
- `outside` has the same index as `s`, so it can be used to select rows of the
  table `s` came from.
- Flagging changes nothing. Removing is a separate decision; write down how
  many rows it took and why.

---

<a id="C"></a>
## C — One summary per group

| You want | Write this |
|---|---|
| one number per group | `table.groupby("g")["x"].mean()` |
| several numbers per group | `table.groupby("g")["x"].agg(["count", "mean", "median", "std"])` |
| named columns | `table.groupby("g").agg(n=("x", "count"), typical=("x", "median"))` |
| a quantile per group | `table.groupby("g")["x"].quantile(0.25)` |
| your own function per group | `table.groupby("g")["x"].agg(my_function)` |
| each group in turn | `for name, values in table.groupby("g")["x"]:` |
| groups in a chosen order | `.reindex(ORDER)` on the result |
| groups by size | `.sort_values(ascending=False)` on the result |

In [ ]:
by_day = cart.groupby("weekday")["cups"].agg(["count", "mean", "median", "std"])
by_day.reindex(ORDER)

In [ ]:
# A loop over the groups, when each group needs more than one line of work.
rows = []
for name, values in cart.groupby("weekend")["cups"]:
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    rows.append({"weekend": name, "count": values.count(), "median": values.median(),
                 "iqr": q3 - q1})
pd.DataFrame(rows)

- `groupby` leaves out rows whose group is missing. `groupby("g",
  dropna=False)` keeps them as a group of their own.
- `count` counts values that are not missing; `size` counts rows.
- `std` of a group with one value is `nan`.

---

<a id="D"></a>
## D — Two columns: correlation

| You want | Write this |
|---|---|
| Pearson's r of two columns of one table | `table["x"].corr(table["y"])` |
| r for every pair of numeric columns | `table.corr(numeric_only=True)` |
| r in numpy | `np.corrcoef(x, y)[0, 1]` (no missing values allowed) |
| r per group | `table.groupby("g").apply(lambda rows: rows["x"].corr(rows["y"]))` |

In [ ]:
print(cart["temperature"].corr(cart["cups"]))
cart[["temperature", "cups", "weekend"]].corr()

From the definition:

```python
pairs = pd.DataFrame({"x": np.asarray(x), "y": np.asarray(y)}).dropna()
dx = pairs["x"] - pairs["x"].mean()
dy = pairs["y"] - pairs["y"].mean()
r = (dx * dy).sum() / ((dx ** 2).sum() * (dy ** 2).sum()) ** 0.5
```

### The index trap

pandas matches two Series **by index label**, not by position. Two columns of
the same table share their labels, so nothing goes wrong. Two Series from
different places may not:

In [ ]:
x = pd.Series([1.0, 2.0, 3.0, 4.0])
y = pd.Series([2.0, 4.0, 1.0, 5.0], index=[10, 11, 12, 13])     # same rows, other labels

print("x.corr(y)                   ", x.corr(y))
print("by position, with np.asarray", pd.Series(np.asarray(x)).corr(pd.Series(np.asarray(y))))

No error, only `nan`, because no label of `x` was found in `y`. The same
happens with `x + y`, and with `pd.DataFrame({"x": x, "y": y})`, which gives
eight half-empty rows. When two Series are meant row for row, take the labels
off with `np.asarray(...)` or `.to_numpy()` or `.reset_index(drop=True)`.

### What r does and does not say

| r | Means | Does not mean |
|---|---|---|
| near 1 or −1 | the points lie close to a straight line | that one causes the other |
| near 0 | no *straight-line* relationship | no relationship: a curve or a cluster can give 0 |
| any value | — | that there is no single point producing it; draw the scatter plot |

r is `nan` when either column does not vary, or when fewer than two complete
rows exist.

---

<a id="E"></a>
## E — Making a figure

| You want | Write this |
|---|---|
| a Figure with one Axes | `fig, ax = plt.subplots()` |
| a chosen size, in inches | `plt.subplots(figsize=(8, 4))` |
| a row of three Axes | `fig, axes = plt.subplots(1, 3)` → `axes[0]`, `axes[1]`, `axes[2]` |
| a 2 × 2 grid | `fig, axes = plt.subplots(2, 2)` → `axes[row, column]` |
| always a 2-D grid, whatever the shape | `plt.subplots(rows, columns, squeeze=False)` |
| the same x or y range in all | `plt.subplots(1, 3, sharex=True, sharey=True)` |
| every Axes of a grid in turn | `for ax in axes.flat:` |
| every Axes of any Figure | `fig.axes` (a list) |
| less overlap between Axes | `fig.tight_layout()` |
| a title above all Axes | `fig.suptitle("...")` |

- **Figure**: the whole picture, the thing that is saved.
- **Axes**: one plot, with an x axis and a y axis. (The name is unfortunate:
  an Axes is not the plural of axis. `ax.xaxis` and `ax.yaxis` are those.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
print(type(axes), axes.shape)

axes[0].hist(cart.loc[~cart["weekend"], "cups"].dropna(), bins=12)
axes[1].hist(cart.loc[cart["weekend"], "cups"].dropna(), bins=12)
axes[0].set_title("weekdays")
axes[1].set_title("weekends")
axes[0].set_ylabel("days")
for ax in axes:
    ax.set_xlabel("cups sold")
fig.tight_layout()

**The shape of `axes` changes with the grid**, which breaks code that works
for one layout and is then used for another:

| Call | `axes` is |
|---|---|
| `plt.subplots()` | one Axes |
| `plt.subplots(1, 3)` | a 1-D array of 3 |
| `plt.subplots(2, 2)` | a 2-D array |
| `plt.subplots(1, n, squeeze=False)` | always 2-D: `axes[0]` is the row, even for `n = 1` |

### Functions that draw

Take the Axes as the first argument, draw on it, return it. The caller decides
where it goes.

```python
def draw_cups(ax, values, title):
    ax.hist(values.dropna(), bins=12)
    ax.set_xlabel("cups sold")
    ax.set_ylabel("days")
    ax.set_title(title)
    return ax
```

Inside such a function, never call `plt.subplots()`, `plt.xlabel(...)` or
`plt.title(...)`: they act on "the current Axes", which may not be the one you
were given.

---

<a id="F"></a>
## F — One chart per kind of question

| The question | Kind | Method | Section |
|---|---|---|---|
| How are the values spread out? | distribution | `ax.hist`, `ax.boxplot` | F1, F5 |
| Which group has more? | comparison | `ax.bar`, `ax.barh` | F2 |
| Do two things go together? | relationship | `ax.scatter` | F3 |
| How did it change over time? | trend | `ax.plot` | F4 |

### F1 — Histogram

| You want | Write this |
|---|---|
| a histogram with 20 bins | `ax.hist(values, bins=20)` |
| bins with edges you choose | `ax.hist(values, bins=[0, 100, 200, 300, 400])` |
| edges evenly spaced on a log axis | `bins=np.geomspace(low, high, n + 1)` and `ax.set_xscale("log")` |
| white lines between the bars | `edgecolor="white"` |
| shares in place of counts | `weights=np.ones(len(values)) / len(values)` |
| a vertical line at a value | `ax.axvline(x, linestyle="--", label="...")` |
| a shaded range | `ax.axvspan(low, high, alpha=0.2)` |
| the counts and edges without drawing | `counts, edges = np.histogram(values, bins=20)` |

`ax.hist` does not accept missing values gracefully: pass `values.dropna()`.

In [ ]:
values = cart["cups"].dropna()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(values, bins=15, color="steelblue", edgecolor="white")
ax.axvline(values.mean(), color="firebrick", linestyle="--", label="mean {:.0f}".format(values.mean()))
ax.axvline(values.median(), color="black", linestyle=":", label="median {:.0f}".format(values.median()))
ax.axvspan(values.quantile(0.25), values.quantile(0.75), color="gold", alpha=0.25,
           label="middle half")
ax.set_xlabel("cups sold in a day")
ax.set_ylabel("days")
ax.set_title("Cups sold per day")
ax.legend()
fig.tight_layout()

The number of bins is a choice. Too few hides the shape, too many shows noise.
Try several; 10 to 40 suits most data. On a **logarithmic** axis:

In [ ]:
skewed = pd.Series(np.random.default_rng(1).lognormal(mean=1.0, sigma=1.6, size=500))
edges = np.geomspace(skewed.min(), skewed.max(), 20 + 1)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(skewed, bins=edges, color="steelblue", edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("value (logarithmic scale)")
ax.set_ylabel("count")
ax.set_title("Bin edges from np.geomspace")
fig.tight_layout()

`np.geomspace(low, high, n + 1)` returns `n + 1` edges, starting exactly at
`low` and ending exactly at `high`, each the same multiple of the one before.
(`np.logspace` takes *exponents* and can miss the end values by a rounding
error, which drops the smallest and largest data point from the histogram.)
Values of 0 or less must be removed first.

### F2 — Bars

| You want | Write this |
|---|---|
| vertical bars | `ax.bar(names, heights)` |
| horizontal bars (long names) | `ax.barh(names, widths)` |
| bars at positions 0, 1, 2 with names | `ax.bar(range(n), heights)`, `ax.set_xticks(range(n))`, `ax.set_xticklabels(names)` |
| the axis from zero | `ax.set_ylim(bottom=0)` (`ax.set_xlim(left=0)` for `barh`) |
| the largest at the top of a `barh` | pass the values in ascending order, or `ax.invert_yaxis()` |
| slanted tick labels | `ax.tick_params(axis="x", labelrotation=45)` |
| the number above each bar | `ax.bar_label(bars, fmt="%.0f")` with `bars = ax.bar(...)` |
| error bars | `ax.bar(names, heights, yerr=spread)` |

In [ ]:
by_day = cart.groupby("weekday")["cups"].median().reindex(ORDER)

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(range(len(by_day)), by_day.values, color="steelblue")
ax.set_xticks(range(len(by_day)))
ax.set_xticklabels(by_day.index)
ax.bar_label(bars, fmt="%.0f")
ax.set_ylim(bottom=0)
ax.set_xlabel("day of the week")
ax.set_ylabel("median cups sold")
ax.set_title("Median cups sold by weekday")
ax.tick_params(axis="x", labelrotation=30)
fig.tight_layout()

Compute the number per group **first** (Section C), then draw. Sort by height
unless the groups have a natural order. A bar's length is the number: the
value axis starts at zero.

### F3 — Scatter plot

| You want | Write this |
|---|---|
| one point per row | `ax.scatter(x, y)` |
| see-through points, where many overlap | `alpha=0.3` |
| smaller points | `s=10` |
| colour by a category | one `ax.scatter` call per group, each with a `label` |
| colour by a number | `ax.scatter(x, y, c=values, cmap="viridis")`, then `fig.colorbar(points)` |
| r in the legend | `label="r = {:.2f}".format(r)` then `ax.legend()` |
| text at a position | `ax.text(x, y, "note")` |
| a straight line through the cloud | `slope, intercept = np.polyfit(x, y, 1)` then `ax.plot(xs, slope * xs + intercept)` |

In [ ]:
complete = cart.dropna(subset=["temperature", "cups"])

fig, ax = plt.subplots(figsize=(7, 3.8))
for is_weekend, colour, name in [(False, "steelblue", "weekday"), (True, "darkorange", "weekend")]:
    rows = complete[complete["weekend"] == is_weekend]
    r = rows["temperature"].corr(rows["cups"])
    ax.scatter(rows["temperature"], rows["cups"], color=colour, alpha=0.7,
               label="{} (r = {:.2f})".format(name, r))
ax.set_xlabel("temperature at noon (°C)")
ax.set_ylabel("cups sold")
ax.set_title("Colder days sell more, on weekdays and weekends alike")
ax.legend()
fig.tight_layout()

`ax.scatter` leaves out a point when either coordinate is missing.

### F4 — Line

| You want | Write this |
|---|---|
| a line through points | `ax.plot(x, y)` |
| markers on the points | `ax.plot(x, y, marker="o")` |
| points only, no line | `ax.plot(x, y, "o")` |
| two lines | two `ax.plot` calls, each with a `label` |
| a horizontal reference line | `ax.axhline(value, linestyle="--")` |
| a smoothed line | `ax.plot(x, y.rolling(7, center=True).mean())` |
| readable date labels | `fig.autofmt_xdate()` |
| a filled band | `ax.fill_between(x, low, high, alpha=0.2)` |

**`ax.plot` joins the points in the order of the rows.** Sort by x first, or
the line runs back and forth. A missing y value breaks the line in two at that
point.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(cart["date"], cart["cups"], color="lightsteelblue", label="each day")
ax.plot(cart["date"], cart["cups"].rolling(7, center=True, min_periods=4).mean(),
        color="steelblue", linewidth=2.5, label="7-day mean")
ax.set_xlabel("date")
ax.set_ylabel("cups sold")
ax.set_title("Cups sold per day (two days were not recorded)")
ax.legend()
fig.autofmt_xdate()

Counts per period for a trend line:

```python
per_month = table["date"].dt.to_period("M").value_counts().sort_index()
ax.plot(per_month.index.to_timestamp(), per_month.values)

per_year = table["date"].dropna().dt.year.value_counts().sort_index()
ax.plot(per_year.index, per_year.values, marker="o")
```

Leave out a first or last period that is incomplete.

### F5 — Box plot

A box plot draws Section B: a box from `q1` to `q3`, a line at the median,
whiskers to the last value inside each fence, and a point for every value
beyond.

| You want | Write this |
|---|---|
| one box | `ax.boxplot(values.dropna())` |
| one box per group | `ax.boxplot([group_1, group_2, ...])` then `ax.set_xticklabels(names)` |
| horizontal boxes | `vert=False` |
| no points for the outliers | `showfliers=False` |

In [ ]:
groups = [cart.loc[cart["weekday"] == day, "cups"].dropna() for day in ORDER]

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.boxplot(groups)
ax.set_xticklabels([day[:3] for day in ORDER])
ax.set_xlabel("day of the week")
ax.set_ylabel("cups sold")
ax.set_title("Cups sold by weekday: median, quartiles and range")
fig.tight_layout()

---

<a id="G"></a>
## G — Labels, titles, legends, text

| You want | Write this |
|---|---|
| a label on the x axis | `ax.set_xlabel("area (acres)")` |
| a label on the y axis | `ax.set_ylabel("records")` |
| a title | `ax.set_title("...")` |
| several at once | `ax.set(xlabel="...", ylabel="...", title="...")` |
| a legend | give each thing a `label="..."`, then `ax.legend()` |
| the legend somewhere else | `ax.legend(loc="upper left")` |
| a legend title | `ax.legend(title="group")` |
| text at data coordinates | `ax.text(x, y, "note")` |
| text at a fixed place in the Axes | `ax.text(0.02, 0.95, "note", transform=ax.transAxes, va="top")` |
| an arrow to a point | `ax.annotate("note", xy=(x, y), xytext=(x2, y2), arrowprops={"arrowstyle": "->"})` |
| larger text | `fontsize=12` on any of them |
| a number formatted in text | `"mean {:,.1f}".format(value)` or `f"mean {value:,.1f}"` |

Formatting numbers for labels:

| Format | 1234.5678 becomes |
|---|---|
| `"{:.2f}"` | `1234.57` |
| `"{:,.0f}"` | `1,235` |
| `"{:.3g}"` | `1.23e+03` |
| `"{:.1%}"` (of 0.1234) | `12.3%` |

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(cart["date"], cart["cups"], color="steelblue")

busiest = cart.loc[cart["cups"].idxmax()]
ax.annotate("busiest day: {:.0f} cups".format(busiest["cups"]),
            xy=(busiest["date"], busiest["cups"]),
            xytext=(cart["date"].iloc[5], busiest["cups"] - 10),
            arrowprops={"arrowstyle": "->"})
ax.text(0.99, 0.04, "source: the cart's own tally", transform=ax.transAxes,
        ha="right", fontsize=8, color="gray")
ax.set(xlabel="date", ylabel="cups sold", title="One summer at the coffee cart")
fig.autofmt_xdate()

What makes a label good:

- **It names the quantity and the unit**: "area of the infestation (acres)",
  not "area", and never the raw column name `total_area`.
- **A title says what to see**: "Weekends sell a third more", not "Cups by
  weekday".
- A legend is for two or more things. One line needs no legend, unless the
  label carries a number such as r.

---

<a id="H"></a>
## H — Scales, limits, ticks

| You want | Write this |
|---|---|
| a logarithmic x axis | `ax.set_xscale("log")` |
| a logarithmic y axis | `ax.set_yscale("log")` |
| the y axis from zero | `ax.set_ylim(bottom=0)` |
| both ends of an axis | `ax.set_ylim(0, 100)`, `ax.set_xlim(low, high)` |
| what the limits are now | `ax.get_ylim()` → `(low, high)` |
| ticks where you choose | `ax.set_xticks([0, 10, 20])` |
| ticks with your own text | `ax.set_xticks(positions)` then `ax.set_xticklabels(names)` |
| rotate tick labels | `ax.tick_params(axis="x", labelrotation=45)` |
| a grid | `ax.grid(True, alpha=0.3)` |
| thousands separators on an axis | `ax.yaxis.set_major_formatter("{x:,.0f}")` |
| percent on an axis | `from matplotlib.ticker import PercentFormatter`; `ax.yaxis.set_major_formatter(PercentFormatter(1.0))` |
| the same range in several Axes | `plt.subplots(..., sharex=True, sharey=True)` |
| equal units on both axes | `ax.set_aspect("equal")` |

In [ ]:
fig, (plain, logged) = plt.subplots(1, 2, figsize=(10, 3.2))
for ax in (plain, logged):
    ax.scatter(skewed[:250], skewed[250:], alpha=0.5, color="steelblue")
    ax.set_xlabel("first half")
    ax.set_ylabel("second half")
logged.set_xscale("log")
logged.set_yscale("log")
plain.set_title("Linear axes: everything in one corner")
logged.set_title("Logarithmic axes")
fig.tight_layout()

When to use which:

| Situation | Axis |
|---|---|
| values spanning several powers of ten; mean far from median | logarithmic, and say so in the label |
| a bar chart | linear, from zero |
| a line showing change | linear; need not start at zero, but then state the size of the change in words |
| panels to be compared | shared |
| values of 0 or less on a log axis | not possible: remove them, or plot `value + 1` and say so |

---

<a id="I"></a>
## I — Colours, markers, line styles

| You want | Write this |
|---|---|
| a named colour | `color="steelblue"`, `"firebrick"`, `"darkorange"`, `"gray"`, `"black"` |
| the default colour cycle | `"C0"`, `"C1"`, `"C2"`, ... |
| see-through | `alpha=0.5` |
| a line style | `linestyle="-"`, `"--"`, `":"`, `"-."` |
| a line width | `linewidth=2` |
| a marker | `marker="o"`, `"s"`, `"^"`, `"x"`, `"."` |
| marker size | `markersize=4` in `ax.plot`; `s=20` in `ax.scatter` |
| the outline of bars | `edgecolor="white"` |

Use colour to carry information, not decoration:

- One variable, one colour. Seven bars in seven colours say that the seven
  differ in some second way, and they do not.
- Use a second colour to pick out the one thing the title talks about.
- About one reader in twelve cannot tell red from green. Vary line style or
  marker as well as colour (`"--"` and `":"` above), and a figure also
  survives a black-and-white printer.

In [ ]:
by_day = cart.groupby("weekday")["cups"].mean().reindex(ORDER)
colours = ["darkorange" if day in ("Saturday", "Sunday") else "lightgray" for day in ORDER]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(range(7), by_day.values, color=colours)
ax.set_xticks(range(7))
ax.set_xticklabels([day[:3] for day in ORDER])
ax.set_ylim(bottom=0)
ax.set(xlabel="day of the week", ylabel="mean cups sold",
       title="Weekends sell about a third more")
fig.tight_layout()

---

<a id="J"></a>
## J — Saving and closing

| You want | Write this |
|---|---|
| save as PNG | `fig.savefig("name.png")` |
| sharper | `fig.savefig("name.png", dpi=150)` |
| no clipped labels | `fig.savefig("name.png", bbox_inches="tight")` |
| a format that scales (for print) | `fig.savefig("name.pdf")` or `"name.svg"` |
| make the folder first | `os.makedirs(folder, exist_ok=True)` |
| a path inside a folder | `os.path.join(folder, name + ".png")` |
| close one figure | `plt.close(fig)` |
| close all figures | `plt.close("all")` |
| which figures are open | `plt.get_fignums()` |

In [ ]:
import os
import tempfile

folder = os.path.join(tempfile.mkdtemp(), "figures")
os.makedirs(folder, exist_ok=True)

figures = {}
for name, column in [("cups", "cups"), ("temperature", "temperature")]:
    fig, ax = plt.subplots()
    ax.hist(cart[column].dropna(), bins=12)
    ax.set(xlabel=column, ylabel="days", title=column)
    figures[name] = fig

for name, fig in figures.items():
    fig.savefig(os.path.join(folder, name + ".png"), dpi=120, bbox_inches="tight")
    plt.close(fig)

print(sorted(os.listdir(folder)), "| figures still open:", plt.get_fignums())

### Where figures appear

| Where the code runs | What happens | What to do |
|---|---|---|
| a notebook cell | open figures are shown when the cell ends, then closed | nothing; `plt.close(fig)` if you do *not* want one shown |
| a program in a Terminal on the cluster | nothing is shown: there is no screen | `fig.savefig(...)`, then `plt.close(fig)` |
| a test | drawn in memory | the test reads the Axes (Section K) |

**Do not call `plt.show()` in a program.** On a machine with no screen it
shows nothing; on a machine with one it stops the program until someone closes
the window. A function that makes a figure returns it or saves it.

matplotlib keeps every figure in memory until it is closed. A loop that makes
figures without closing them gets a warning after twenty and runs out of
memory later.

A Figure variable as the **last line** of a notebook cell is displayed like
any other value. That is how to look at a figure a function returned:
`fig = make_figure(...)` in one cell, then `fig` alone at the end of a cell.

---

<a id="K"></a>
## K — Asking an Axes what is on it

An Axes remembers what was drawn. A program can check a figure without anyone
looking at it.

| You want to know | Ask |
|---|---|
| the labels and title | `ax.get_xlabel()`, `ax.get_ylabel()`, `ax.get_title()` |
| whether anything is drawn | `ax.has_data()` |
| the scale of an axis | `ax.get_xscale()`, `ax.get_yscale()` → `"linear"` or `"log"` |
| the limits | `ax.get_xlim()`, `ax.get_ylim()` → `(low, high)` |
| the bars (and histogram bars) | `ax.patches`; each has `get_x()`, `get_width()`, `get_height()` |
| the lines | `ax.lines`; each has `get_xdata()`, `get_ydata()`, `get_label()` |
| the scattered points | `ax.collections[0].get_offsets()` → rows of (x, y) |
| the text placed with `ax.text` | `ax.texts`; each has `get_text()` |
| the legend entries | `ax.get_legend()` (None when there is none), then `.get_texts()` |
| the tick labels | `[t.get_text() for t in ax.get_xticklabels()]`, after `fig.canvas.draw()` |
| all Axes of a figure | `fig.axes` |

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["a", "b", "c"], [50, 52, 55])
ax.set_ylim(48, 56)
ax.set_xlabel("group")

print("x label  ", repr(ax.get_xlabel()))
print("y label  ", repr(ax.get_ylabel()))
print("title    ", repr(ax.get_title()))
print("has data ", ax.has_data())
print("bars     ", len(ax.patches), [p.get_height() for p in ax.patches])
print("y scale  ", ax.get_yscale(), " y from", ax.get_ylim()[0])
fig.tight_layout()

That printout is an audit: no y label, no title, and bars on an axis that
starts at 48. `"".strip()` and `"   ".strip()` are both empty, which is the
test for "no label".

```python
for number, ax in enumerate(fig.axes):
    if not ax.get_title().strip():
        print("Axes", number, "has no title")
```

---

<a id="L"></a>
## L — The pandas shortcut

A Series or DataFrame can draw itself. It uses matplotlib underneath, returns
the Axes, and accepts `ax=` to draw on one you made.

| You want | Write this |
|---|---|
| a quick histogram | `s.plot.hist(bins=20, ax=ax)` or `s.hist(bins=20, ax=ax)` |
| a quick bar chart of a Series | `s.plot.bar(ax=ax)`, `s.plot.barh(ax=ax)` |
| a quick line (index on x) | `s.plot(ax=ax)` |
| a quick scatter | `table.plot.scatter(x="a", y="b", ax=ax)` |
| a quick box plot per column | `table.plot.box(ax=ax)` |

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
returned = cart.groupby("weekday")["cups"].mean().reindex(ORDER).plot.bar(ax=ax, color="steelblue")
print(returned is ax)

ax.set(xlabel="day of the week", ylabel="mean cups sold", title="The same chart, drawn by pandas")
ax.tick_params(axis="x", labelrotation=30)
fig.tight_layout()

Good for looking at data quickly. Two things to remember: **always pass
`ax=ax`** (without it pandas draws on the current Axes or makes a new figure),
and the labels are still your job.

---

<a id="M"></a>
## M — Checklist and common mistakes

### Before anyone else sees a figure

| Check | A program can test it |
|---|---|
| Both axes labelled | yes |
| Labels name the unit | no |
| A title, and it says what to see | that there is one: yes |
| Bars start at zero | yes |
| A logarithmic axis is announced in its label | partly |
| The first and last period of a trend are complete | no |
| Categories mean the same thing throughout | no |
| Panels to be compared share their axes | yes |
| The summary number quoted beside it is the right one (mean or median?) | no |

### Messages you will meet

| Message or symptom | Cause | Fix |
|---|---|---|
| `nan` from `np.mean`, `np.var`, `np.quantile` | a missing value in the array | `s.dropna()`, the pandas method, or `np.nanmean` |
| `nan` from `x.corr(y)` with good data | the two Series have different index labels | `np.asarray(...)` on both (Section D) |
| a variance that differs from the expected one in the second digit | count versus count − 1 | `ddof` (Section A) |
| an empty figure in a notebook | the figure was made in one cell and drawn on in another | create and draw in the same cell |
| two copies of a figure in a notebook | the cell made a figure *and* ends with `fig` | drop the last line |
| `<matplotlib...>` printed under a figure | the last line returned an object | end the cell with `fig.tight_layout()` or a `;` |
| a line that zigzags across the picture | rows not sorted by x | `sort_values` first |
| a histogram on a log axis with bars of wildly different widths | equal-width bins on a log axis | `bins=np.geomspace(...)` |
| `ValueError: Data has no positive values, and therefore cannot be log-scaled` | zeros or negatives on a log axis | remove them first |
| `'numpy.ndarray' object has no attribute 'hist'` | `axes` is an array: `plt.subplots(1, 3)` | `axes[0].hist(...)` |
| `'Axes' object is not subscriptable` | `plt.subplots()` gave one Axes and the code wrote `axes[0]` | `squeeze=False` |
| `RuntimeWarning: More than 20 figures have been opened` | figures never closed | `plt.close(fig)` after saving |
| `UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown` | `plt.show()` where there is no screen | save the figure; remove `show()` |
| tick labels overlapping | long names on a vertical bar chart | `ax.barh`, or `labelrotation=45` |
| labels cut off in the saved file | the figure's margins | `bbox_inches="tight"` |
| the plot went onto the wrong Axes | `plt.xlabel` / `plt.title` inside a function | `ax.set_xlabel` / `ax.set_title` |

### From pyplot to the object interface

Code you find elsewhere is often written with `plt.` only. The translation:

| pyplot | object interface |
|---|---|
| `plt.figure(figsize=(8, 4))` | `fig, ax = plt.subplots(figsize=(8, 4))` |
| `plt.hist(x)`, `plt.bar(...)`, `plt.scatter(...)`, `plt.plot(...)` | `ax.hist(x)`, `ax.bar(...)`, `ax.scatter(...)`, `ax.plot(...)` |
| `plt.xlabel("...")`, `plt.ylabel("...")`, `plt.title("...")` | `ax.set_xlabel("...")`, `ax.set_ylabel("...")`, `ax.set_title("...")` |
| `plt.xlim(a, b)`, `plt.ylim(a, b)` | `ax.set_xlim(a, b)`, `ax.set_ylim(a, b)` |
| `plt.xscale("log")` | `ax.set_xscale("log")` |
| `plt.xticks(rotation=45)` | `ax.tick_params(axis="x", labelrotation=45)` |
| `plt.legend()` | `ax.legend()` |
| `plt.subplot(1, 2, 1)` | `fig, axes = plt.subplots(1, 2)` then `axes[0]` |
| `plt.savefig("f.png")` | `fig.savefig("f.png")` |
| `plt.show()` | nothing in a notebook; `fig.savefig(...)` in a program |

The only `plt.` calls this course uses are `plt.subplots`, `plt.close` and
`plt.get_fignums`.